### Ноутбук "EDA и подготовка данных"

#### Описание

Генерация и загрузка синтетических данных в PostgreSQL, первичная проверка целостности перед основным анализом.

##### Импорт модулей и библиотек

In [1]:
import sys
sys.path.append("../src")

from generate_data import generate_users, generate_subscription, generate_payments, generate_ab_assignments

In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import text

from db_connection import get_engine
engine = get_engine()

##### Создание таблицы "users"

Генерация датафрейма "users_df" с 3000 пользователей, без столбца "id"

In [3]:
users_df = generate_users(3000)
display(users_df.head())
display(users_df.info())

,acquisition_channel,plan,country,signup_date
0,email,free,Russia,2026-03-23
1,paid_search,free,Belarus,2025-05-10
2,social,free,China,2026-06-16
3,organic,basic,Belarus,2026-01-18
4,organic,free,Belarus,2026-05-21


<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   acquisition_channel  3000 non-null   str   
 1   plan                 3000 non-null   str   
 2   country              3000 non-null   str   
 3   signup_date          3000 non-null   object
dtypes: object(1), str(3)
memory usage: 93.9+ KB


None

*Очистка таблицы "users"*

In [4]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE users RESTART IDENTITY CASCADE;"))

Генерация датафрейма "users_db" со столбцом "id"

In [5]:
users_df.to_sql("users", engine, if_exists="append", index=False)

users_db = pd.read_sql("SELECT * FROM users;", engine)
display(users_db.columns)
display(users_db.shape)

Index(['id', 'acquisition_channel', 'plan', 'country', 'signup_date'], dtype='str')

(3000, 5)

##### Создание таблицы "subscription"

Генерация датафрейма "subscriptions_df" без столбца "id"

In [6]:
subscriptions_df = generate_subscription(users_db)
display(subscriptions_df.head())
display(subscriptions_df.info())

,user_id,plan,price,start_date,end_date,status
0,4,basic,100.0,2026-01-20,NaT,active
1,10,basic,100.0,2026-04-25,NaT,active
2,12,basic,100.0,2026-01-23,NaT,active
3,14,basic,100.0,2026-07-02,NaT,active
4,15,basic,100.0,2026-01-17,NaT,active


<class 'pandas.DataFrame'>
RangeIndex: 1387 entries, 0 to 1386
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     1387 non-null   int64         
 1   plan        1387 non-null   str           
 2   price       1387 non-null   float64       
 3   start_date  1387 non-null   datetime64[us]
 4   end_date    422 non-null    datetime64[us]
 5   status      1387 non-null   str           
dtypes: datetime64[us](2), float64(1), int64(1), str(2)
memory usage: 65.1 KB


None

Генерация датафрейма "subscriptions_db" со столбцом "id"

In [7]:
subscriptions_df.to_sql("subscriptions", engine, if_exists="append", index=False)

subscriptions_db = pd.read_sql("SELECT * FROM subscriptions;", engine)
display(subscriptions_db.columns)
display(subscriptions_db.shape)

Index(['id', 'user_id', 'plan', 'price', 'start_date', 'end_date', 'status'], dtype='str')

(1387, 7)

##### Создание таблицы "payments"

Генерация датафрейма "payments_df" без столбца "id"

In [8]:
payments_df = generate_payments(subscriptions_db)
display(payments_df.head())
display(payments_df.info())

,subscription_id,amount,payment_date,status
0,1,100.0,2026-01-20,succeeded
1,1,100.0,2026-02-19,succeeded
2,1,100.0,2026-03-21,succeeded
3,1,100.0,2026-04-20,failed
4,1,100.0,2026-05-20,succeeded


<class 'pandas.DataFrame'>
RangeIndex: 11229 entries, 0 to 11228
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   subscription_id  11229 non-null  int64         
 1   amount           11229 non-null  float64       
 2   payment_date     11229 non-null  datetime64[us]
 3   status           11229 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 351.0 KB


None

Генерация датафрейма "payments_df" со столбцом "id"

In [9]:
payments_df.to_sql("payments", engine, if_exists="append", index=False)

payments_db = pd.read_sql("SELECT * FROM payments;", engine)
display(payments_db.columns)
display(payments_db.shape)

Index(['id', 'subscription_id', 'amount', 'payment_date', 'status'], dtype='str')

(11229, 5)

##### Создание таблицы "ab_test_assignments"

Генерация датафрейма "assignments_df" без столбца "id"

In [10]:
assignments_df = generate_ab_assignments(users_db)
display(assignments_df.head())
display(assignments_df.info())

,user_id,test,variant,assigned_at
0,14,onboarding,A,2026-07-02
1,14,pricing,B,2026-07-04
2,17,onboarding,A,2026-08-07
3,17,pricing,A,2026-08-05
4,18,onboarding,A,2026-07-22


<class 'pandas.DataFrame'>
RangeIndex: 1036 entries, 0 to 1035
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      1036 non-null   int64         
 1   test         1036 non-null   str           
 2   variant      1036 non-null   str           
 3   assigned_at  1036 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 32.5 KB


None

Генерация датафрейма "assignments_df" со столбцом "id"

In [11]:
assignments_df.to_sql("ab_test_assignments", engine, if_exists="append", index=False)

assignments_db = pd.read_sql("SELECT * FROM ab_test_assignments;", engine)
display(assignments_db.columns)
display(assignments_db.shape)

Index(['id', 'user_id', 'test', 'variant', 'assigned_at'], dtype='str')

(1036, 5)

In [12]:
for table in ['users', 'subscriptions', 'payments', 'events', 'ab_test_assignments']:
    count = pd.read_sql(f"SELECT COUNT(*) FROM {table};", engine)
    print(table, count.iloc[0,0])

users 3000
subscriptions 1387
payments 11229
events 0
ab_test_assignments 1036
